# OKRs 2026Q3

In [105]:
# hide-output
# Import libraries and initialise the BigQuery connector

# show → code input visible by default
# hide-output → output hidden by default
# show hide-output → both (can combine on one line)

# Standard data analysis stack + project utilities
import pandas as pd
from common_lib.sql import BigQueryConnector
from common_lib.export import export_notebook_html
import plotly.express as px
import numpy as np
import plotly.graph_objects as go

bqc = BigQueryConnector()

## Period comparison parameters

Shared by every section's period-over-period comparison (last subsection of each analysis) — defined once here so both periods stay in sync across the whole notebook.

In [106]:
import datetime as dt

period1_start, period1_end = dt.date(2025, 7, 1), dt.date(2025, 9, 30)   # Period 1: Jul-Sep 2025 (Q3)
period2_start, period2_end = dt.date(2026, 1, 1), dt.date(2026, 8, 31)  # Period 2: Jan-Jul 2026

print(f"Period 1: {period1_start} -> {period1_end}")
print(f"Period 2: {period2_start} -> {period2_end}")

Period 1: 2025-07-01 -> 2025-09-30
Period 2: 2026-01-01 -> 2026-08-31


## Get data

### Return rates

In [107]:
# Compute symmetric A/B date windows of equal length anchored on 2026-06-01 (FTUE launch date)

import datetime as dt

start_date = dt.date(2025, 1, 1)
end_date = dt.date(2026, 8, 20)

In [108]:
# hide-output
# Toggle: True re-queries BigQuery and overwrites local cache, False loads from pickle
refresh_data = True

In [109]:
# hide-output
# Estimate query cost for player level + game day SQL before executing

query_location = './sql/returnrate.sql'
parameters = {
    'start_date': start_date.strftime('%Y-%m-%d'),
    'end_date': end_date.strftime('%Y-%m-%d'), 
    'loyalty_segments': [],  # empty = no filter, all segments
}

# Print cost estimate before running — avoids accidental expensive queries (~$1.63 / 242 GB)
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 3.75 GB when run.
Estimated query cost: $0.03


In [110]:
# hide-output
# Fetch player level + game day data from BigQuery or load from local pickle cache
data = pd.DataFrame()

# Toggle to True to re-run the BQ query and overwrite the local cache

if refresh_data:
    data = bqc.get(query='./sql/returnrate.sql', is_path=True, query_parameters=parameters)
    data.to_pickle('./data/returnrate.pkl')
else:
    # Load from local cache to avoid repeated query costs
    data = pd.read_pickle('./data/returnrate.pkl')

In [111]:
data.sort_values(by=['dt'], inplace=True)
data

,dt,dau,returnrate_day_01,dau_d1,returnrate_day_03,dau_d3,returnrate_day_07,dau_d7,returnrate_day_14,dau_d14,...,returnrate_within_01,dau_w1,returnrate_within_03,dau_w3,returnrate_within_07,dau_w7,returnrate_within_14,dau_w14,returnrate_within_28,dau_w28
144,2025-01-01,185977,0.836383,155548,0.803470,149427,0.766132,142483,0.725380,134904,...,0.836383,155548,0.917995,170726,0.943095,175394,0.953397,177310,0.960802,178687
410,2025-01-02,191294,0.834077,159554,0.807260,154424,0.766731,146671,0.725464,138777,...,0.834077,159554,0.918095,175626,0.942899,180371,0.953898,182475,0.961300,183891
35,2025-01-03,190754,0.833414,158977,0.806783,153897,0.767664,146435,0.724588,138218,...,0.833414,158977,0.920730,175633,0.944583,180183,0.955131,182195,0.962538,183608
418,2025-01-04,190089,0.844420,160515,0.803992,152830,0.770355,146436,0.723340,137499,...,0.844420,160515,0.921952,175253,0.945578,179744,0.955947,181715,0.963233,183100
487,2025-01-05,193029,0.830896,160387,0.794679,153396,0.774640,149528,0.731393,141180,...,0.830896,160387,0.912661,176170,0.941967,181827,0.953365,184027,0.961348,185568
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
401,2026-08-16,62772,0.832983,52288,0.814933,51155,NaN,0,NaN,0,...,0.832983,52288,0.915647,57477,NaN,58118,NaN,58118,NaN,58118
368,2026-08-17,62890,0.847130,53276,0.819033,51509,NaN,0,NaN,0,...,0.847130,53276,0.924201,58123,NaN,58123,NaN,58123,NaN,58123
479,2026-08-18,63117,0.848741,53570,NaN,0,NaN,0,NaN,0,...,0.848741,53570,NaN,57182,NaN,57182,NaN,57182,NaN,57182
336,2026-08-19,63103,0.843700,53240,NaN,0,NaN,0,NaN,0,...,0.843700,53240,NaN,53240,NaN,53240,NaN,53240,NaN,53240


#### Return rates — Dedicated segment only

In [112]:
# hide-output
# Toggle: True re-queries BigQuery and overwrites local cache, False loads from pickle
# Separate from the main refresh_data toggle above so backfilling this cache doesn't
# needlessly re-run the (already cached) unfiltered return rate / loyalty queries too.
refresh_data_dedicated = True

# Estimate query cost for the dedicated-segment-only return rate SQL before executing

query_location = './sql/returnrate.sql'
parameters_dedicated = {
    'start_date': start_date.strftime('%Y-%m-%d'),
    'end_date': end_date.strftime('%Y-%m-%d'),
    'loyalty_segments': ['1. 26-28 (dedicated)'],
}

cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters_dedicated)

This query will process 4.68 GB when run.
Estimated query cost: $0.03


In [113]:
# hide-output
# Fetch return rate data filtered to the dedicated loyalty segment, or load from local pickle cache
data_dedicated = pd.DataFrame()

if refresh_data_dedicated:
    data_dedicated = bqc.get(query='./sql/returnrate.sql', is_path=True, query_parameters=parameters_dedicated)
    data_dedicated.to_pickle('./data/returnrate_dedicated.pkl')
else:
    # Load from local cache to avoid repeated query costs
    data_dedicated = pd.read_pickle('./data/returnrate_dedicated.pkl')

In [114]:
data_dedicated.sort_values(by=['dt'], inplace=True)
data_dedicated

,dt,dau,returnrate_day_01,dau_d1,returnrate_day_03,dau_d3,returnrate_day_07,dau_d7,returnrate_day_14,dau_d14,...,returnrate_within_01,dau_w1,returnrate_within_03,dau_w3,returnrate_within_07,dau_w7,returnrate_within_14,dau_w14,returnrate_within_28,dau_w28
516,2025-01-01,88611,0.967013,85688,0.940109,83304,0.898263,79596,0.832222,73744,...,0.967013,85688,0.984133,87205,0.985036,87285,0.985871,87359,0.987090,87467
60,2025-01-02,89109,0.968634,86314,0.942610,83995,0.897923,80013,0.832969,74225,...,0.968634,86314,0.984850,87759,0.985579,87824,0.986421,87899,0.987835,88025
453,2025-01-03,89189,0.965668,86127,0.943054,84110,0.895424,79862,0.830629,74083,...,0.965668,86127,0.983933,87756,0.984763,87830,0.985974,87938,0.987308,88057
352,2025-01-04,88968,0.968730,86186,0.943317,83925,0.894715,79601,0.826646,73545,...,0.968730,86186,0.985557,87683,0.986523,87769,0.987366,87844,0.988738,87966
581,2025-01-05,89170,0.967904,86308,0.942111,84008,0.895738,79873,0.828709,73896,...,0.967904,86308,0.983907,87735,0.984860,87820,0.985645,87890,0.986688,87983
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
476,2026-08-16,35143,0.966195,33955,0.947642,33303,NaN,0,NaN,0,...,0.966195,33955,0.983866,34576,NaN,34585,NaN,34585,NaN,34585
535,2026-08-17,35160,0.971018,34141,0.946189,33268,NaN,0,NaN,0,...,0.971018,34141,0.986206,34675,NaN,34675,NaN,34675,NaN,34675
548,2026-08-18,35137,0.972166,34159,NaN,0,NaN,0,NaN,0,...,0.972166,34159,NaN,34610,NaN,34610,NaN,34610,NaN,34610
406,2026-08-19,35059,0.969052,33974,NaN,0,NaN,0,NaN,0,...,0.969052,33974,NaN,33974,NaN,33974,NaN,33974,NaN,33974


### User segments

In [115]:
# hide-output
# Estimate query cost for player level + game day SQL before executing

query_location = './sql/activity.sql'
parameters = {
    'start_date': start_date.strftime('%Y-%m-%d'),
    'end_date': end_date.strftime('%Y-%m-%d'), 
    'exclude_networks': []
}

# Print cost estimate before running — avoids accidental expensive queries (~$1.63 / 242 GB)
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 9.44 GB when run.
Estimated query cost: $0.06


In [116]:
# hide-output
# Fetch player level + game day data from BigQuery or load from local pickle cache
loyalty_data = pd.DataFrame()

# Toggle to True to re-run the BQ query and overwrite the local cache

if refresh_data:
    loyalty_data = bqc.get(query='./sql/activity.sql', is_path=True, query_parameters=parameters)
    loyalty_data.to_pickle('./data/activity.pkl')
else:
    # Load from local cache to avoid repeated query costs
    loyalty_data = pd.read_pickle('./data/activity.pkl')

In [117]:
loyalty_data

,user_id,dt,dt_week,dt_month,install_dt,install_dt_week,install_dt_month,days_since_install,loyalty_segment,dsi_segment,usd_net_iap_revenue,usd_net_ad_revenue
0,3A42F036F1E1ABCA,2026-08-03,2026-08-02,2026-08-01,2026-07-12,2026-07-12,2026-07-01,22,0.0 (new install),D007-D027,NaN,NaN
1,56B4431C6DC2D4BD,2025-05-23,2025-05-18,2025-05-01,2025-04-07,2025-04-06,2025-04-01,46,2. 19-25 (frequent),D028-D090,NaN,NaN
2,72F3A1BE19B7CF25,2025-09-09,2025-09-07,2025-09-01,2025-08-29,2025-08-24,2025-08-01,11,0.0 (new install),D007-D027,NaN,NaN
3,F94A80F365D6420F,2025-05-08,2025-05-04,2025-05-01,2023-01-17,2023-01-15,2023-01-01,842,2. 19-25 (frequent),D364+,NaN,NaN
4,8EA165221ABDD8E9,2025-03-08,2025-03-02,2025-03-01,2021-12-24,2021-12-19,2021-12-01,1170,1. 26-28 (dedicated),D364+,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
75304362,933AD3496A00CB69,2026-07-29,2026-07-26,2026-07-01,2023-12-03,2023-12-03,2023-12-01,969,1. 26-28 (dedicated),D364+,NaN,0.035299
75304363,18BB1FDCD106D7E9,2026-07-14,2026-07-12,2026-07-01,2024-12-29,2024-12-29,2024-12-01,562,2. 19-25 (frequent),D364+,NaN,NaN
75304364,D8F59A5BBEC6AE39,2025-05-03,2025-04-27,2025-05-01,2025-04-06,2025-04-06,2025-04-01,27,2. 19-25 (frequent),D007-D027,NaN,1.143746
75304365,E4CEDABC0D34C24A,2026-01-15,2026-01-11,2026-01-01,2024-07-24,2024-07-21,2024-07-01,540,3. 04-18 (moderate),D364+,0.927289,0.261924


# Return rates

In [118]:
returnrate = data.copy()
returnrate_dedicated = data_dedicated.copy()

In [119]:
# Create a line chart showing return rates over time
fig = px.line(
    returnrate,
    x='dt',
    y=['returnrate_day_01', 'returnrate_day_03', 'returnrate_day_07', 'returnrate_day_14', 'returnrate_day_28'],
    title='Return Rates Over Time',
    labels={'dt': 'Date', 'value': 'Return Rate', 'variable': 'Days'},
    markers=False,
    height=600,
    width=1500
)

#fig.update_layout(hovermode='x unified')
fig.show()

In [120]:
# Create a line chart showing return rates over time
fig = px.line(
    returnrate,
    x='dt',
    y=['returnrate_within_01', 'returnrate_within_03', 'returnrate_within_07', 'returnrate_within_14', 'returnrate_within_28'],
    title='Return Rates within Over Time',
    labels={'dt': 'Date', 'value': 'Return Rate', 'variable': 'Days'},
    markers=False,
    height=600,
    width=1500
)

fig.show()

In [121]:
# Create monthly average of returnrate_within_28
returnrate_monthly = returnrate[['dt', 'returnrate_within_28']].copy()
returnrate_monthly['dt'] = pd.to_datetime(returnrate_monthly['dt'])
returnrate_monthly['year_month'] = returnrate_monthly['dt'].dt.to_period('M')
returnrate_monthly_avg = returnrate_monthly.groupby('year_month')['returnrate_within_28'].mean().reset_index()
returnrate_monthly_avg.columns = ['Month', 'Avg Return Rate (28-day)']

returnrate_monthly_avg['Month'] = returnrate_monthly_avg['Month'].astype(str)

returnrate_monthly_avg['churn'] = 1 - returnrate_monthly_avg['Avg Return Rate (28-day)']

fig = px.line(
    returnrate_monthly_avg,
    x='Month',
    y='churn',
    title='Monthly Average 28-Day Churn Rate',
    labels={'Month': 'Month', 'churn': 'Churn Rate'},
    height=500,
    width=1200,
)
fig.show()

In [122]:
# Compare dedicated segment return rates: 2025 vs 2026 averages per horizon, with relative difference

returnrate_years = returnrate.copy()
returnrate_years['dt'] = pd.to_datetime(returnrate_years['dt'])
returnrate_years['year'] = returnrate_years['dt'].dt.year

within_cols = ['returnrate_within_01', 'returnrate_within_03', 'returnrate_within_07', 'returnrate_within_14', 'returnrate_within_28']

yearly_avg = returnrate_years.groupby('year')[within_cols].mean().T
yearly_avg.columns = [str(c) for c in yearly_avg.columns]

# Relative difference (2026 vs 2025)
if '2025' in yearly_avg.columns and '2026' in yearly_avg.columns:
    yearly_avg['rel_diff'] = (yearly_avg['2026'] - yearly_avg['2025']) / yearly_avg['2025']

yearly_avg.index.name = 'metric'
yearly_avg = yearly_avg.reset_index()
print(yearly_avg)

for col in within_cols:
    row = yearly_avg[yearly_avg['metric'] == col].iloc[0]
    rel_diff_pct = row['rel_diff'] * 100

    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=['2025', '2026'],
        y=[row['2025'], row['2026']],
        text=[f"{row['2025']:.4f}", f"{row['2026']:.4f}"],
        textposition='outside',
        marker_color=['#636efa', '#EF553B'],
    ))
    fig.update_layout(
        title=f"Return Rates — {col}: 2025 vs 2026 avg (relative diff: {rel_diff_pct:+.2f}%)",
        yaxis_title=col,
        xaxis_title='Year',
        width=800,
        height=350,
    )
    #fig.show()

                 metric      2025      2026  rel_diff
0  returnrate_within_01  0.820450  0.829144  0.010596
1  returnrate_within_03  0.906199  0.910197  0.004412
2  returnrate_within_07  0.934335  0.938447  0.004401
3  returnrate_within_14  0.946617  0.953446  0.007214
4  returnrate_within_28  0.954265  0.961843  0.007941


## Dedicated

In [123]:
# Create a line chart showing return rates over time
fig = px.line(
    returnrate_dedicated,
    x='dt',
    y=['returnrate_within_01', 'returnrate_within_03', 'returnrate_within_07', 'returnrate_within_14', 'returnrate_within_28'],
    title='Return Rates Within Over Time',
    labels={'dt': 'Date', 'value': 'Return Rate', 'variable': 'Days'},
    markers=False,
    height=600,
    width=1500
)

fig.show()

In [124]:
# Compare dedicated segment return rates: 2025 vs 2026 averages per horizon, with relative difference

returnrate_dedicated_years = returnrate_dedicated.copy()
returnrate_dedicated_years['dt'] = pd.to_datetime(returnrate_dedicated_years['dt'])
returnrate_dedicated_years['year'] = returnrate_dedicated_years['dt'].dt.year

within_cols = ['returnrate_within_01', 'returnrate_within_03', 'returnrate_within_07', 'returnrate_within_14', 'returnrate_within_28']

yearly_avg = returnrate_dedicated_years.groupby('year')[within_cols].mean().T
yearly_avg.columns = [str(c) for c in yearly_avg.columns]

# Relative difference (2026 vs 2025)
if '2025' in yearly_avg.columns and '2026' in yearly_avg.columns:
    yearly_avg['rel_diff'] = (yearly_avg['2026'] - yearly_avg['2025']) / yearly_avg['2025']

yearly_avg.index.name = 'metric'
yearly_avg = yearly_avg.reset_index()
print(yearly_avg)

for col in within_cols:
    row = yearly_avg[yearly_avg['metric'] == col].iloc[0]
    rel_diff_pct = row['rel_diff'] * 100

    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=['2025', '2026'],
        y=[row['2025'], row['2026']],
        text=[f"{row['2025']:.4f}", f"{row['2026']:.4f}"],
        textposition='outside',
        marker_color=['#636efa', '#EF553B'],
    ))
    fig.update_layout(
        title=f"Dedicated Segment — {col}: 2025 vs 2026 avg (relative diff: {rel_diff_pct:+.2f}%)",
        yaxis_title=col,
        xaxis_title='Year',
        width=800,
        height=350,
    )
    #fig.show()

                 metric      2025      2026  rel_diff
0  returnrate_within_01  0.964102  0.967634  0.003663
1  returnrate_within_03  0.983360  0.985196  0.001867
2  returnrate_within_07  0.984354  0.986159  0.001834
3  returnrate_within_14  0.985345  0.987092  0.001772
4  returnrate_within_28  0.986447  0.988106  0.001681


In [125]:
# Create monthly average of returnrate_within_28
returnrate_dedicated_monthly = returnrate_dedicated[['dt', 'returnrate_within_28']].copy()
returnrate_dedicated_monthly['dt'] = pd.to_datetime(returnrate_dedicated_monthly['dt'])
returnrate_dedicated_monthly['year_month'] = returnrate_dedicated_monthly['dt'].dt.to_period('M')
returnrate_dedicated_monthly_avg = returnrate_dedicated_monthly.groupby('year_month')['returnrate_within_28'].mean().reset_index()
returnrate_dedicated_monthly_avg.columns = ['Month', 'Avg Return Rate (28-day)']

returnrate_dedicated_monthly_avg['Month'] = returnrate_dedicated_monthly_avg['Month'].astype(str)

returnrate_dedicated_monthly_avg['churn'] = 1 - returnrate_dedicated_monthly_avg['Avg Return Rate (28-day)']

fig = px.line(
    returnrate_dedicated_monthly_avg,
    x='Month',
    y='churn',
    title='Monthly Average 28-Day Churn Rate',
    labels={'Month': 'Month', 'churn': 'Churn Rate'},
    height=500,
    width=1200,
)
fig.show()

## RRW Forecast & Scenario Analysis

Everything below targets a single metric column — change `metric_col` to retarget any other return-rate horizon (e.g. `returnrate_within_07`).

1. Decompose into trend / weekly seasonal / residual, check stationarity
2. Characterize variance and flag outliers robustly (MAD-based, not std/mean)
3. Detect regime shifts with unsupervised change-point detection (PELT)
4. Fit a 3-month forecast with uncertainty bands, backtested
5. Summary chart + table
6. Standalone "what if return rate got a +X% uplift" scenario overlay

## 1. Data prep + trend/seasonality decomposition

In [126]:
# hide-output
from aux_functions import (
    decompose_series, flag_robust_outliers, detect_changepoints, select_training_start,
    fit_rate_forecast, apply_uplift_scenario, compare_periods,
    plot_stl_decomposition, plot_outliers, plot_changepoints, plot_forecast_summary, plot_uplift_scenario,
    plot_period_comparison,
)

### Parameters

In [127]:
# --- All tunable parameters for this section, in one place ---

metric_col = 'returnrate_within_28'    # which RRW horizon to analyze — swap to retarget (e.g. 'returnrate_within_07')

outlier_threshold = 3.5                # Sec 2: modified z-score threshold for flagging outliers (Iglewicz & Hoaglin standard)
min_segment_days = 14                  # Sec 3: minimum days between two detected change-points (regimes)
min_training_days = 90                 # Sec 4: forecast training window — walk back through regimes until >= this many days
sarimax_order = (1, 1, 1)              # Sec 4: SARIMAX (p, d, q) non-seasonal order
sarimax_seasonal_order = (1, 1, 1, 7)  # Sec 4: SARIMAX seasonal order, period=7 (weekly)
forecast_horizon_days = 90             # Sec 4: how many days ahead to forecast
backtest_days = 28                     # Sec 4: holdout days for the pre-forecast backtest
ci_alpha = 0.20                        # Sec 4: forecast interval width (0.20 -> 80% interval, ~P10/P90)
uplift_pct = 0.005                     # Sec 6: relative uplift applied to the forecast curve (e.g. 0.02 = +2%)

In [128]:
# hide-output
rrw = returnrate[['dt', metric_col]].dropna().copy()
rrw['dt'] = pd.to_datetime(rrw['dt'])
rrw = rrw.set_index('dt').asfreq('D')
n_gaps = rrw[metric_col].isna().sum()
rrw[metric_col] = rrw[metric_col].interpolate()  # fill any single-day calendar gaps

rrw_series = rrw[metric_col]
rrw_series.name = metric_col

print(f"{metric_col}: {len(rrw)} days, {rrw.index.min().date()} -> {rrw.index.max().date()} ({n_gaps} interpolated gap-days)")

returnrate_within_28: 569 days, 2025-01-01 -> 2026-07-23 (0 interpolated gap-days)


In [129]:
# hide-output
rrw_decomp, rrw_adf_stat, rrw_adf_p = decompose_series(rrw_series, period=7)
print(f"ADF test on STL residual: stat={rrw_adf_stat:.3f}, p={rrw_adf_p:.4f} -> {'stationary' if rrw_adf_p < 0.05 else 'NOT stationary'}")

ADF test on STL residual: stat=-7.716, p=0.0000 -> stationary


In [130]:
# Chart — STL decomposition: observed+trend / weekly seasonal / residual
plot_stl_decomposition(rrw_decomp, title=f'{metric_col} — STL decomposition (weekly seasonality)').show()

## 2. Variance & outliers

Variance is measured on the STL residual (raw series variance is dominated by trend/seasonality, which isn't "noise"). Outliers are flagged with a MAD-based robust z-score rather than std/mean, so a handful of extreme days can't inflate the very threshold used to catch them.

**In plain terms:**

- **MAD (Median Absolute Deviation)** is a robust way to measure "how spread out" the data is. It's computed in two steps, both using the *median* rather than the *mean*: (1) find the median of the residual, (2) for every point, measure its distance from that median, then take the median of *those* distances. That's the MAD.
- **Why median instead of mean/std?** The usual way to measure spread — standard deviation around the mean — has a circularity problem: a few extreme values pull the mean toward them and inflate the standard deviation, which makes the extreme values look *less* extreme relative to that inflated yardstick. The median and MAD barely move when a handful of points are extreme, because the median only cares about the middle-ranked value, not the size of the outliers. That's what "robust" means here.
- **The modified z-score** turns "distance from the median" into a standardized score, so it reads on roughly the same scale as a familiar z-score (e.g. "3.5 SDs away"): `0.6745 × (value − median) / MAD`. The `0.6745` constant is there so that, for normally-distributed data, this number lines up with an ordinary z-score — it's just a unit conversion, not a tunable knob.
- **The 3.5 threshold** is a standard rule of thumb (Iglewicz & Hoaglin): a modified z-score beyond ±3.5 flags a point as very unlikely to belong to the same distribution as the bulk of the data. It's a convention, not a law — worth adjusting if it flags too much or too little for this series.

In [131]:
# hide-output
rrw_decomp['modified_z'], rrw_decomp['is_outlier'] = flag_robust_outliers(rrw_decomp['resid'], threshold=outlier_threshold)

cov = rrw_decomp['resid'].std() / rrw_decomp['value'].mean()
print(f"Residual std: {rrw_decomp['resid'].std():.5f}  |  MAD: {(rrw_decomp['resid'] - rrw_decomp['resid'].median()).abs().median():.5f}  |  series mean: {rrw_decomp['value'].mean():.4f}")
print(f"Coefficient of variation (resid std / series mean): {cov:.3%}")
print(f"Outliers flagged (|modified z| > {outlier_threshold}): {rrw_decomp['is_outlier'].sum()} / {len(rrw_decomp)} days")
print()
print("Note: flagged days often cluster in time rather than appearing as isolated spikes — those clusters")
print("usually indicate a regime shift (see change-point section below) rather than independent anomalies.")

rrw_decomp[rrw_decomp['is_outlier']][['value', 'resid', 'modified_z']]

Residual std: 0.00253  |  MAD: 0.00063  |  series mean: 0.9570
Coefficient of variation (resid std / series mean): 0.264%
Outliers flagged (|modified z| > 3.5): 59 / 569 days

Note: flagged days often cluster in time rather than appearing as isolated spikes — those clusters
usually indicate a regime shift (see change-point section below) rather than independent anomalies.


,value,resid,modified_z
dt,,,
2025-01-28,0.946626,-0.010712,-11.532168
2025-01-29,0.941654,-0.016699,-17.968982
2025-01-30,0.946720,-0.008653,-9.317932
2025-02-07,0.942233,-0.013074,-14.071889
2025-02-08,0.945987,-0.006738,-7.259145
2025-02-24,0.962129,0.004554,4.882371
2025-02-25,0.963730,0.006607,7.089481
2025-02-28,0.946253,-0.009596,-10.332039
2025-03-01,0.935331,-0.016629,-17.894160


In [132]:
# Chart — observed series with flagged outlier days marked
plot_outliers(rrw_decomp, rrw_decomp['is_outlier'], title=f'{metric_col} with robust-outlier days flagged').show()

## 3. Change-point detection (regime shifts)

Unsupervised structural-break detection (PELT, `ruptures`) on the deseasonalized series — no prior list of "known events" required. Detects level shifts (a permanent-looking step, e.g. a feature launch) rather than single-day spikes, which is what the outlier check above already covers. The series is standardized first since the penalty is scale-sensitive; `changepoint_penalty` defaults to the standard BIC-style `log(n)` — raise it for fewer/more conservative breakpoints, lower it for more sensitivity.

In [133]:
# hide-output
changepoint_dates, changepoint_penalty = detect_changepoints(rrw_decomp['deseasonalized'], min_segment_days=min_segment_days)

print(f"Penalty: {changepoint_penalty:.2f}  |  Change-points detected: {len(changepoint_dates)}")
for d in changepoint_dates:
    print(' ', d.date())

Penalty: 6.34  |  Change-points detected: 10
  2025-01-25
  2025-02-09
  2025-02-24
  2025-05-15
  2025-08-18
  2025-09-02
  2025-12-11
  2025-12-26
  2026-03-26
  2026-06-14


In [134]:
# Chart — observed series with detected change-points marked
plot_changepoints(rrw_decomp, changepoint_dates, title=f'{metric_col} with detected regime changes (dashed lines)').show()

## 4. Forecast (3 months ahead)

**Training window**: the most recent detected regime is often too short to fit weekly seasonality reliably on its own (here, only ~18 days since the last change-point) — so we walk backward through the detected regimes and use the most recent *set* of them that together provide at least `min_training_days`, rather than an arbitrary flat lookback or the single latest (possibly too-short) regime.

**Model**: SARIMAX with weekly seasonality, fit on the **logit-transformed** series rather than the raw rate — this keeps forecast bands naturally within (0, 1). A plain fit on the raw proportion produced a nonsensical >100% upper bound at the 90-day horizon; the logit transform is the standard fix for forecasting bounded rates.

**Backtest**: hold out the last 28 days of the training window, refit, and check forecast error before trusting the live 90-day extrapolation.

In [135]:
# hide-output
# Walk back through detected regimes until we have >= min_training_days (set in Parameters above)
training_start = select_training_start(rrw_decomp.index, changepoint_dates, min_training_days=min_training_days)
train = rrw_series.loc[training_start:]
print(f"Training window: {training_start.date()} -> {rrw_decomp.index.max().date()} ({len(train)} days)")

Training window: 2026-03-26 -> 2026-07-23 (120 days)


In [136]:
# hide-output
backtest_mae, backtest_mape, forecast_df = fit_rate_forecast(
    train, order=sarimax_order, seasonal_order=sarimax_seasonal_order,
    horizon=forecast_horizon_days, backtest_days=backtest_days, ci_alpha=ci_alpha,
)
print(f"Backtest (last {backtest_days} days of training window): MAE={backtest_mae:.5f}  MAPE={backtest_mape:.3%}")

Backtest (last 28 days of training window): MAE=0.00322  MAPE=0.335%


In [137]:
# hide-output
print(f"Forecast horizon: {forecast_df.index.min().date()} -> {forecast_df.index.max().date()}")
for h in [30, 60, 90]:
    row = forecast_df.iloc[h - 1]
    print(f"  +{h}d ({forecast_df.index[h-1].date()}): {row['forecast']:.4f}  [{row['low']:.4f}, {row['high']:.4f}]")

Forecast horizon: 2026-07-24 -> 2026-10-21
  +30d (2026-08-22): 0.9521  [0.9378, 0.9632]
  +60d (2026-09-21): 0.9484  [0.9170, 0.9684]
  +90d (2026-10-21): 0.9439  [0.8853, 0.9735]


## 5. Summary: history + forecast

In [138]:
# Chart — training window history + 90-day forecast with 80% band
plot_forecast_summary(
    rrw_series, train, forecast_df, training_start,
    title=f'{metric_col} — history + {forecast_horizon_days}-day forecast', ci_alpha=ci_alpha,
).show()

In [139]:
# Summary table — forecast at key checkpoints
summary_table = forecast_df.iloc[[29, 59, 89]].reset_index().rename(columns={
    'dt': 'Date', 'forecast': 'P50 (forecast)', 'low': 'P10', 'high': 'P90',
})
summary_table.insert(0, 'Days out', [30, 60, 90])
summary_table[['P10', 'P50 (forecast)', 'P90']] = summary_table[['P10', 'P50 (forecast)', 'P90']].round(4)
summary_table

,Days out,Date,P50 (forecast),P10,P90
0,30,2026-08-22,0.9521,0.9378,0.9632
1,60,2026-09-21,0.9484,0.9170,0.9684
2,90,2026-10-21,0.9439,0.8853,0.9735


## 6. Scenario: what if return rate gets a % uplift

Standalone overlay on the forecast curve itself — applies a relative uplift directly to the forecasted rate (clipped at 100%, since it's a probability), not a downstream DAU/cohort simulation. Adjust `uplift_pct` to try different scenarios.

In [140]:
# hide-output
scenario_df = apply_uplift_scenario(forecast_df, uplift_pct)

print(f"Uplift scenario: {uplift_pct:+.1%} relative")
for h in [30, 60, 90]:
    base = scenario_df['forecast'].iloc[h - 1]
    up = scenario_df['forecast_uplift'].iloc[h - 1]
    print(f"  +{h}d: baseline={base:.4f}  uplifted={up:.4f}  (delta={up - base:+.4f})")

Uplift scenario: +0.5% relative
  +30d: baseline=0.9521  uplifted=0.9568  (delta=+0.0048)
  +60d: baseline=0.9484  uplifted=0.9532  (delta=+0.0047)
  +90d: baseline=0.9439  uplifted=0.9486  (delta=+0.0047)


In [141]:
# Chart — baseline forecast vs uplift scenario
plot_uplift_scenario(
    rrw_series, scenario_df, uplift_pct,
    title=f'{metric_col} — baseline vs {uplift_pct:+.1%} uplift scenario', lookback_days=60,
).show()

## 7. Period comparison: Period 1 vs Period 2

Average `metric_col` within each of the two shared periods defined at the top of the notebook, and the relative uplift between them.

In [142]:
# hide-output
rrw_period_comparison = compare_periods(rrw_series, period1_start, period1_end, period2_start, period2_end)

print(f"Period 1 ({period1_start} -> {period1_end}, n={rrw_period_comparison['period1_n']} days): avg {metric_col} = {rrw_period_comparison['period1_mean']:.4f}")
print(f"Period 2 ({period2_start} -> {period2_end}, n={rrw_period_comparison['period2_n']} days): avg {metric_col} = {rrw_period_comparison['period2_mean']:.4f}")
print(f"Uplift: {rrw_period_comparison['abs_diff']:+.4f}  ({rrw_period_comparison['pct_uplift']:+.1%} relative)")

Period 1 (2025-07-01 -> 2025-09-30, n=92 days): avg returnrate_within_28 = 0.9544
Period 2 (2026-01-01 -> 2026-08-31, n=204 days): avg returnrate_within_28 = 0.9618
Uplift: +0.0075  (+0.8% relative)


In [143]:
# Chart — Period 1 vs Period 2 average
plot_period_comparison(
    rrw_period_comparison, title=f'{metric_col}: Period 1 vs Period 2 average', y_label=metric_col,
).show()

# Loyalty Segments

In [144]:
loyalty_data

,user_id,dt,dt_week,dt_month,install_dt,install_dt_week,install_dt_month,days_since_install,loyalty_segment,dsi_segment,usd_net_iap_revenue,usd_net_ad_revenue
0,3A42F036F1E1ABCA,2026-08-03,2026-08-02,2026-08-01,2026-07-12,2026-07-12,2026-07-01,22,0.0 (new install),D007-D027,NaN,NaN
1,56B4431C6DC2D4BD,2025-05-23,2025-05-18,2025-05-01,2025-04-07,2025-04-06,2025-04-01,46,2. 19-25 (frequent),D028-D090,NaN,NaN
2,72F3A1BE19B7CF25,2025-09-09,2025-09-07,2025-09-01,2025-08-29,2025-08-24,2025-08-01,11,0.0 (new install),D007-D027,NaN,NaN
3,F94A80F365D6420F,2025-05-08,2025-05-04,2025-05-01,2023-01-17,2023-01-15,2023-01-01,842,2. 19-25 (frequent),D364+,NaN,NaN
4,8EA165221ABDD8E9,2025-03-08,2025-03-02,2025-03-01,2021-12-24,2021-12-19,2021-12-01,1170,1. 26-28 (dedicated),D364+,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
75304362,933AD3496A00CB69,2026-07-29,2026-07-26,2026-07-01,2023-12-03,2023-12-03,2023-12-01,969,1. 26-28 (dedicated),D364+,NaN,0.035299
75304363,18BB1FDCD106D7E9,2026-07-14,2026-07-12,2026-07-01,2024-12-29,2024-12-29,2024-12-01,562,2. 19-25 (frequent),D364+,NaN,NaN
75304364,D8F59A5BBEC6AE39,2025-05-03,2025-04-27,2025-05-01,2025-04-06,2025-04-06,2025-04-01,27,2. 19-25 (frequent),D007-D027,NaN,1.143746
75304365,E4CEDABC0D34C24A,2026-01-15,2026-01-11,2026-01-01,2024-07-24,2024-07-21,2024-07-01,540,3. 04-18 (moderate),D364+,0.927289,0.261924


In [145]:
loyalty_agg = loyalty_data[['dt', 'loyalty_segment', 'user_id']][loyalty_data.loyalty_segment!='0.0 (new install)'].copy()

loyalty_agg = loyalty_agg.groupby(['dt','loyalty_segment']).agg(
    unique_users = ('user_id', 'nunique'),
).reset_index().sort_values(by=['dt','loyalty_segment'])

# Add total unique users per day
loyalty_agg['total_users_per_day'] = loyalty_agg.groupby('dt')['unique_users'].transform('sum')

# Add percentage column
loyalty_agg['percentage'] = (loyalty_agg['unique_users'] / loyalty_agg['total_users_per_day'])

loyalty_agg

,dt,loyalty_segment,unique_users,total_users_per_day,percentage
0,2025-01-01,1. 26-28 (dedicated),88523,156763,0.564693
1,2025-01-01,2. 19-25 (frequent),37491,156763,0.239157
2,2025-01-01,3. 04-18 (moderate),25878,156763,0.165077
3,2025-01-01,4. 01-03 (infrequent),4871,156763,0.031072
4,2025-01-02,1. 26-28 (dedicated),89028,161031,0.552862
...,...,...,...,...,...
2383,2026-08-19,4. 01-03 (infrequent),2258,60406,0.037380
2384,2026-08-20,1. 26-28 (dedicated),34801,59883,0.581150
2385,2026-08-20,2. 19-25 (frequent),13240,59883,0.221098
2386,2026-08-20,3. 04-18 (moderate),9565,59883,0.159728


In [146]:
# Create a line chart showing return rates over time
fig = px.line(
    loyalty_agg,
    x='dt',
    y=['percentage'],
    color='loyalty_segment',
    title='Loyalty Segment Unique Users Over Time',
    labels={'dt': 'Date', 'value': 'Unique Users', 'variable': 'Loyalty Segment'},
    markers=False,
    height=600,
    width=1500
)

fig.show()

## Loyalty Segment Forecast & Scenario Analysis

Same 6-step pipeline as the RRW section above (same `aux_functions.py` helpers), applied to a loyalty segment's daily share of DAU. Change `segment_value` to retarget any other segment (e.g. `'2. 19-25 (frequent)'`, `'3. 04-18 (moderate)'`, `'4. 01-03 (infrequent)'`).

## 1. Data prep + trend/seasonality decomposition

### Parameters

In [147]:
# --- All tunable parameters for this section, in one place ---

segment_value = '1. 26-28 (dedicated)'  # which loyalty segment to analyze — swap to retarget (e.g. '2. 19-25 (frequent)')

outlier_threshold = 3.5                # Sec 2: modified z-score threshold for flagging outliers (Iglewicz & Hoaglin standard)
min_segment_days = 14                  # Sec 3: minimum days between two detected change-points (regimes)
min_training_days = 90                 # Sec 4: forecast training window — walk back through regimes until >= this many days
sarimax_order = (1, 1, 1)              # Sec 4: SARIMAX (p, d, q) non-seasonal order
sarimax_seasonal_order = (1, 1, 1, 7)  # Sec 4: SARIMAX seasonal order, period=7 (weekly)
forecast_horizon_days = 90             # Sec 4: how many days ahead to forecast
backtest_days = 28                     # Sec 4: holdout days for the pre-forecast backtest
ci_alpha = 0.20                        # Sec 4: forecast interval width (0.20 -> 80% interval, ~P10/P90)
uplift_pct_loyalty = 0.005              # Sec 6: relative uplift applied to the forecast curve (e.g. 0.05 = +5%)

In [148]:
# hide-output
loyalty_seg = loyalty_agg[loyalty_agg['loyalty_segment'] == segment_value][['dt', 'percentage']].dropna().copy()
loyalty_seg['dt'] = pd.to_datetime(loyalty_seg['dt'])
loyalty_seg = loyalty_seg.set_index('dt').asfreq('D')
n_gaps = loyalty_seg['percentage'].isna().sum()
loyalty_seg['percentage'] = loyalty_seg['percentage'].interpolate()  # fill any single-day calendar gaps

loyalty_series = loyalty_seg['percentage']
loyalty_series.name = segment_value

print(f"{segment_value}: {len(loyalty_seg)} days, {loyalty_seg.index.min().date()} -> {loyalty_seg.index.max().date()} ({n_gaps} interpolated gap-days)")

1. 26-28 (dedicated): 597 days, 2025-01-01 -> 2026-08-20 (0 interpolated gap-days)


In [149]:
# hide-output
loyalty_decomp, loyalty_adf_stat, loyalty_adf_p = decompose_series(loyalty_series, period=7)
print(f"ADF test on STL residual: stat={loyalty_adf_stat:.3f}, p={loyalty_adf_p:.4f} -> {'stationary' if loyalty_adf_p < 0.05 else 'NOT stationary'}")

ADF test on STL residual: stat=-12.580, p=0.0000 -> stationary


In [150]:
# Chart — STL decomposition: observed+trend / weekly seasonal / residual
plot_stl_decomposition(loyalty_decomp, title=f'{segment_value} — STL decomposition (weekly seasonality)').show()

## 2. Variance & outliers

Same MAD-based robust z-score as the RRW section (see the plain-language explanation there for how it works).

In [151]:
# hide-output
loyalty_decomp['modified_z'], loyalty_decomp['is_outlier'] = flag_robust_outliers(loyalty_decomp['resid'], threshold=outlier_threshold)

cov = loyalty_decomp['resid'].std() / loyalty_decomp['value'].mean()
print(f"Residual std: {loyalty_decomp['resid'].std():.5f}  |  MAD: {(loyalty_decomp['resid'] - loyalty_decomp['resid'].median()).abs().median():.5f}  |  series mean: {loyalty_decomp['value'].mean():.4f}")
print(f"Coefficient of variation (resid std / series mean): {cov:.3%}")
print(f"Outliers flagged (|modified z| > {outlier_threshold}): {loyalty_decomp['is_outlier'].sum()} / {len(loyalty_decomp)} days")

loyalty_decomp[loyalty_decomp['is_outlier']][['value', 'resid', 'modified_z']]

Residual std: 0.00303  |  MAD: 0.00097  |  series mean: 0.5734
Coefficient of variation (resid std / series mean): 0.529%
Outliers flagged (|modified z| > 3.5): 50 / 597 days


,value,resid,modified_z
dt,,,
2025-01-08,0.554320,-0.010582,-7.373309
2025-01-15,0.556991,-0.008796,-6.135289
2025-02-17,0.578367,-0.006110,-4.273378
2025-02-25,0.591887,0.005716,3.924578
2025-02-26,0.595781,0.007944,5.469233
2025-02-27,0.596318,0.006478,4.452643
2025-05-02,0.578718,0.005439,3.732621
2025-05-23,0.573772,0.009168,6.317573
2025-05-24,0.575557,0.010779,7.434529


In [152]:
# Chart — observed series with flagged outlier days marked
plot_outliers(loyalty_decomp, loyalty_decomp['is_outlier'], title=f'{segment_value} with robust-outlier days flagged').show()

## 3. Change-point detection (regime shifts)

Same PELT-based unsupervised detection as the RRW section above, on this segment's deseasonalized share.

In [153]:
# hide-output
changepoint_dates_loyalty, changepoint_penalty_loyalty = detect_changepoints(loyalty_decomp['deseasonalized'], min_segment_days=min_segment_days)

print(f"Penalty: {changepoint_penalty_loyalty:.2f}  |  Change-points detected: {len(changepoint_dates_loyalty)}")
for d in changepoint_dates_loyalty:
    print(' ', d.date())

Penalty: 6.39  |  Change-points detected: 8
  2025-01-25
  2025-04-10
  2025-05-25
  2025-06-19
  2025-12-11
  2026-01-25
  2026-02-24
  2026-06-04


In [154]:
# Chart — observed series with detected change-points marked
plot_changepoints(loyalty_decomp, changepoint_dates_loyalty, title=f'{segment_value} with detected regime changes (dashed lines)').show()

## 4. Forecast (3 months ahead)

Same training-window walk-back + logit-space SARIMAX approach as the RRW section above.

In [155]:
# hide-output
# Walk back through detected regimes until we have >= min_training_days (set in Parameters above)
training_start_loyalty = select_training_start(loyalty_decomp.index, changepoint_dates_loyalty, min_training_days=min_training_days)
train_loyalty = loyalty_series.loc[training_start_loyalty:]
print(f"Training window: {training_start_loyalty.date()} -> {loyalty_decomp.index.max().date()} ({len(train_loyalty)} days)")

Training window: 2026-02-24 -> 2026-08-20 (178 days)


In [156]:
# hide-output
backtest_mae_loyalty, backtest_mape_loyalty, forecast_df_loyalty = fit_rate_forecast(
    train_loyalty, order=sarimax_order, seasonal_order=sarimax_seasonal_order,
    horizon=forecast_horizon_days, backtest_days=backtest_days, ci_alpha=ci_alpha,
)
print(f"Backtest (last {backtest_days} days of training window): MAE={backtest_mae_loyalty:.5f}  MAPE={backtest_mape_loyalty:.3%}")

Backtest (last 28 days of training window): MAE=0.01105  MAPE=1.888%


In [157]:
# hide-output
print(f"Forecast horizon: {forecast_df_loyalty.index.min().date()} -> {forecast_df_loyalty.index.max().date()}")
for h in [30, 60, 90]:
    row = forecast_df_loyalty.iloc[h - 1]
    print(f"  +{h}d ({forecast_df_loyalty.index[h-1].date()}): {row['forecast']:.4f}  [{row['low']:.4f}, {row['high']:.4f}]")

Forecast horizon: 2026-08-21 -> 2026-11-18
  +30d (2026-09-19): 0.5874  [0.5619, 0.6125]
  +60d (2026-10-19): 0.5736  [0.5263, 0.6197]
  +90d (2026-11-18): 0.5721  [0.5003, 0.6410]


## 5. Summary: history + forecast

In [158]:
# Chart — training window history + 90-day forecast with 80% band
plot_forecast_summary(
    loyalty_series, train_loyalty, forecast_df_loyalty, training_start_loyalty,
    title=f'{segment_value} — history + {forecast_horizon_days}-day forecast', ci_alpha=ci_alpha,
).show()

In [159]:
# Summary table — forecast at key checkpoints
summary_table_loyalty = forecast_df_loyalty.iloc[[29, 59, 89]].reset_index().rename(columns={
    'dt': 'Date', 'forecast': 'P50 (forecast)', 'low': 'P10', 'high': 'P90',
})
summary_table_loyalty.insert(0, 'Days out', [30, 60, 90])
summary_table_loyalty[['P10', 'P50 (forecast)', 'P90']] = summary_table_loyalty[['P10', 'P50 (forecast)', 'P90']].round(4)
summary_table_loyalty

,Days out,Date,P50 (forecast),P10,P90
0,30,2026-09-19,0.5874,0.5619,0.6125
1,60,2026-10-19,0.5736,0.5263,0.6197
2,90,2026-11-18,0.5721,0.5003,0.6410


## 6. Scenario: what if this segment's share gets a % uplift

Same standalone overlay approach as the RRW section. Adjust `uplift_pct_loyalty` to try different scenarios.

In [160]:
# hide-output
scenario_df_loyalty = apply_uplift_scenario(forecast_df_loyalty, uplift_pct_loyalty)

print(f"Uplift scenario: {uplift_pct_loyalty:+.1%} relative")
for h in [30, 60, 90]:
    base = scenario_df_loyalty['forecast'].iloc[h - 1]
    up = scenario_df_loyalty['forecast_uplift'].iloc[h - 1]
    print(f"  +{h}d: baseline={base:.4f}  uplifted={up:.4f}  (delta={up - base:+.4f})")

Uplift scenario: +0.5% relative
  +30d: baseline=0.5874  uplifted=0.5904  (delta=+0.0029)
  +60d: baseline=0.5736  uplifted=0.5765  (delta=+0.0029)
  +90d: baseline=0.5721  uplifted=0.5750  (delta=+0.0029)


In [161]:
# Chart — baseline forecast vs uplift scenario
plot_uplift_scenario(
    loyalty_series, scenario_df_loyalty, uplift_pct_loyalty,
    title=f'{segment_value} — baseline vs {uplift_pct_loyalty:+.1%} uplift scenario', lookback_days=60,
).show()

## 7. Period comparison: Period 1 vs Period 2

Average `segment_value`'s share within each of the two shared periods defined at the top of the notebook, and the relative uplift between them.

In [162]:
# hide-output
loyalty_period_comparison = compare_periods(loyalty_series, period1_start, period1_end, period2_start, period2_end)

print(f"Period 1 ({period1_start} -> {period1_end}, n={loyalty_period_comparison['period1_n']} days): avg share = {loyalty_period_comparison['period1_mean']:.4f}")
print(f"Period 2 ({period2_start} -> {period2_end}, n={loyalty_period_comparison['period2_n']} days): avg share = {loyalty_period_comparison['period2_mean']:.4f}")
print(f"Uplift: {loyalty_period_comparison['abs_diff']:+.4f}  ({loyalty_period_comparison['pct_uplift']:+.1%} relative)")

Period 1 (2025-07-01 -> 2025-09-30, n=92 days): avg share = 0.5621
Period 2 (2026-01-01 -> 2026-08-31, n=232 days): avg share = 0.5857
Uplift: +0.0236  (+4.2% relative)


In [163]:
# Chart — Period 1 vs Period 2 average
plot_period_comparison(
    loyalty_period_comparison, title=f'{segment_value}: Period 1 vs Period 2 average', y_label=segment_value,
).show()

# ARPDAU

In [164]:
# --- All tunable parameters for this section, in one place ---

segment_value = '1. 26-28 (dedicated)'  # which loyalty segment to analyze — swap to retarget (e.g. '2. 19-25 (frequent)')

outlier_threshold = 7                # Sec 2: modified z-score threshold for flagging outliers (Iglewicz & Hoaglin standard)
min_segment_days = 14                  # Sec 3: minimum days between two detected change-points (regimes)
min_training_days = 90                 # Sec 4: forecast training window — walk back through regimes until >= this many days
sarimax_order = (1, 1, 1)              # Sec 4: SARIMAX (p, d, q) non-seasonal order
sarimax_seasonal_order = (1, 1, 1, 7)  # Sec 4: SARIMAX seasonal order, period=7 (weekly)
forecast_horizon_days = 90             # Sec 4: how many days ahead to forecast
backtest_days = 28                     # Sec 4: holdout days for the pre-forecast backtest
ci_alpha = 0.20                        # Sec 4: forecast interval width (0.20 -> 80% interval, ~P10/P90)
uplift_pct_loyalty = 0.02              # Sec 6: relative uplift applied to the forecast curve (e.g. 0.02 = +2%)

In [165]:
loyalty_data

,user_id,dt,dt_week,dt_month,install_dt,install_dt_week,install_dt_month,days_since_install,loyalty_segment,dsi_segment,usd_net_iap_revenue,usd_net_ad_revenue
0,3A42F036F1E1ABCA,2026-08-03,2026-08-02,2026-08-01,2026-07-12,2026-07-12,2026-07-01,22,0.0 (new install),D007-D027,NaN,NaN
1,56B4431C6DC2D4BD,2025-05-23,2025-05-18,2025-05-01,2025-04-07,2025-04-06,2025-04-01,46,2. 19-25 (frequent),D028-D090,NaN,NaN
2,72F3A1BE19B7CF25,2025-09-09,2025-09-07,2025-09-01,2025-08-29,2025-08-24,2025-08-01,11,0.0 (new install),D007-D027,NaN,NaN
3,F94A80F365D6420F,2025-05-08,2025-05-04,2025-05-01,2023-01-17,2023-01-15,2023-01-01,842,2. 19-25 (frequent),D364+,NaN,NaN
4,8EA165221ABDD8E9,2025-03-08,2025-03-02,2025-03-01,2021-12-24,2021-12-19,2021-12-01,1170,1. 26-28 (dedicated),D364+,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
75304362,933AD3496A00CB69,2026-07-29,2026-07-26,2026-07-01,2023-12-03,2023-12-03,2023-12-01,969,1. 26-28 (dedicated),D364+,NaN,0.035299
75304363,18BB1FDCD106D7E9,2026-07-14,2026-07-12,2026-07-01,2024-12-29,2024-12-29,2024-12-01,562,2. 19-25 (frequent),D364+,NaN,NaN
75304364,D8F59A5BBEC6AE39,2025-05-03,2025-04-27,2025-05-01,2025-04-06,2025-04-06,2025-04-01,27,2. 19-25 (frequent),D007-D027,NaN,1.143746
75304365,E4CEDABC0D34C24A,2026-01-15,2026-01-11,2026-01-01,2024-07-24,2024-07-21,2024-07-01,540,3. 04-18 (moderate),D364+,0.927289,0.261924


In [166]:
arpdau_agg = loyalty_data[['dt', 'user_id', 'usd_net_iap_revenue','usd_net_ad_revenue']].copy()

arpdau_agg = arpdau_agg.groupby(['dt']).agg(
    unique_users = ('user_id', pd.Series.nunique),
    total_iap_revenue = ('usd_net_iap_revenue', 'sum'),
    total_ad_revenue = ('usd_net_ad_revenue', 'sum'),
).reset_index().sort_values(by=['dt'])

arpdau_agg['arpdau_iap'] = (arpdau_agg['total_iap_revenue'] / arpdau_agg['unique_users']).round(4)
arpdau_agg['arpdau_ad'] = (arpdau_agg['total_ad_revenue'] / arpdau_agg['unique_users']).round(4)

arpdau_agg

,dt,unique_users,total_iap_revenue,total_ad_revenue,arpdau_iap,arpdau_ad
0,2025-01-01,184240,38727.305724,15588.401840,0.2102,0.0846
1,2025-01-02,189594,35609.032592,15305.708157,0.1878,0.0807
2,2025-01-03,189065,38767.160679,15391.533088,0.2050,0.0814
3,2025-01-04,188449,42423.292575,16571.630822,0.2251,0.0879
4,2025-01-05,191439,37484.262951,17123.588668,0.1958,0.0894
...,...,...,...,...,...,...
592,2026-08-16,62348,11814.066088,5012.197022,0.1895,0.0804
593,2026-08-17,62491,12457.984035,5428.631085,0.1994,0.0869
594,2026-08-18,62711,13789.840769,5714.682380,0.2199,0.0911
595,2026-08-19,62739,17767.665096,5958.495010,0.2832,0.0950


In [167]:
arpdau_seg_agg = loyalty_data[['dt', 'loyalty_segment', 'user_id', 'usd_net_iap_revenue','usd_net_ad_revenue']].copy()

arpdau_seg_agg = arpdau_seg_agg.groupby(['dt','loyalty_segment']).agg(
    unique_users = ('user_id', 'nunique'),
    total_iap_revenue = ('usd_net_iap_revenue', 'sum'),
    total_ad_revenue = ('usd_net_ad_revenue', 'sum'),
).reset_index().sort_values(by=['dt','loyalty_segment'])

arpdau_seg_agg['total_users_per_day'] = arpdau_seg_agg.groupby('dt')['unique_users'].transform('sum')

arpdau_seg_agg['arpdau_local_iap'] = arpdau_seg_agg['total_iap_revenue'] / arpdau_seg_agg['unique_users']
arpdau_seg_agg['arpdau_local_ad'] = arpdau_seg_agg['total_ad_revenue'] / arpdau_seg_agg['unique_users']

arpdau_seg_agg['arpdau_iap'] = arpdau_seg_agg['total_iap_revenue'] / arpdau_seg_agg['total_users_per_day']
arpdau_seg_agg['arpdau_ad'] = arpdau_seg_agg['total_ad_revenue'] / arpdau_seg_agg['total_users_per_day']

arpdau_seg_agg

,dt,loyalty_segment,unique_users,total_iap_revenue,total_ad_revenue,total_users_per_day,arpdau_local_iap,arpdau_local_ad,arpdau_iap,arpdau_ad
0,2025-01-01,0.0 (new install),27477,7727.672059,2229.679960,184240,0.281241,0.081147,0.041944,0.012102
1,2025-01-01,1. 26-28 (dedicated),88523,21573.657445,8381.265206,184240,0.243707,0.094679,0.117095,0.045491
2,2025-01-01,2. 19-25 (frequent),37491,5124.186204,2657.692628,184240,0.136678,0.070889,0.027813,0.014425
3,2025-01-01,3. 04-18 (moderate),25878,3706.306264,1931.501006,184240,0.143222,0.074639,0.020117,0.010484
4,2025-01-01,4. 01-03 (infrequent),4871,595.483753,388.263040,184240,0.122251,0.079709,0.003232,0.002107
...,...,...,...,...,...,...,...,...,...,...
2980,2026-08-20,0.0 (new install),2316,310.568330,176.052412,62199,0.134097,0.076016,0.004993,0.002830
2981,2026-08-20,1. 26-28 (dedicated),34801,11002.862000,3442.202028,62199,0.316165,0.098911,0.176898,0.055342
2982,2026-08-20,2. 19-25 (frequent),13240,2319.724446,965.910851,62199,0.175206,0.072954,0.037295,0.015529
2983,2026-08-20,3. 04-18 (moderate),9565,1542.473373,846.277526,62199,0.161262,0.088476,0.024799,0.013606


In [168]:
# Create a line chart showing return rates over time
fig = px.line(
    arpdau_agg,
    x='dt',
    y=['arpdau_iap', 'arpdau_ad'],
    title='ARPDAU Over Time',
    labels={'dt': 'Date', 'value': 'ARPDAU', 'variable': 'Revenue Type'},
    markers=False,
    height=600,
    width=1500,
    hover_data={'unique_users': True, 'total_iap_revenue': ':.0f', 'total_ad_revenue': ':.0f'},
)

fig.show()

In [169]:
# Create a line chart showing return rates over time
fig = px.line(
    arpdau_seg_agg[arpdau_seg_agg['loyalty_segment'] == segment_value],
    x='dt',
    y=['arpdau_local_iap', 'arpdau_local_ad', 'arpdau_iap', 'arpdau_ad'],
    title='ARPDAU Over Time by Loyalty Segment',
    labels={'dt': 'Date', 'value': 'ARPDAU', 'variable': 'Loyalty Segment'},
    markers=False,
    height=600,
    width=1500
)

fig.show()

## Core ARPDAU (excluding new installs)

New installs are about to slow down significantly, and day-0 users are close to guaranteed $0 revenue — so as they shrink as a share of DAU, blended ARPDAU will rise mechanically even if nothing about how well existing users are monetized has changed (a mix-shift artifact, not a real improvement). **Core ARPDAU** re-blends revenue and users across segments 1-4 only (excluding `0.0 (new install)`), so it isn't affected by UA volume changes. The 7-section pipelines below run on core ARPDAU; blended ARPDAU (all segments, computed earlier) is kept as a reference chart so any future divergence between the two is itself a diagnostic signal.

In [170]:
# hide-output
from aux_functions import fit_value_forecast

In [171]:
# hide-output
core_arpdau_agg = loyalty_data[loyalty_data['loyalty_segment'] != '0.0 (new install)'][['dt', 'user_id', 'usd_net_iap_revenue', 'usd_net_ad_revenue']].copy()

core_arpdau_agg = core_arpdau_agg.groupby('dt').agg(
    unique_users=('user_id', 'nunique'),
    total_iap_revenue=('usd_net_iap_revenue', 'sum'),
    total_ad_revenue=('usd_net_ad_revenue', 'sum'),
).reset_index().sort_values('dt')
core_arpdau_agg['dt'] = pd.to_datetime(core_arpdau_agg['dt'])

core_arpdau_agg['arpdau_iap'] = core_arpdau_agg['total_iap_revenue'] / core_arpdau_agg['unique_users']
core_arpdau_agg['arpdau_ad'] = core_arpdau_agg['total_ad_revenue'] / core_arpdau_agg['unique_users']

core_arpdau_agg

,dt,unique_users,total_iap_revenue,total_ad_revenue,arpdau_iap,arpdau_ad
0,2025-01-01,156763,30999.633665,13358.721880,0.197748,0.085216
1,2025-01-02,161031,27956.151133,13103.430087,0.173607,0.081372
2,2025-01-03,161127,30452.024698,13119.154688,0.188994,0.081421
3,2025-01-04,160747,33677.306935,14102.531301,0.209505,0.087731
4,2025-01-05,163220,28510.395094,14533.534562,0.174675,0.089043
...,...,...,...,...,...,...
592,2026-08-16,60009,11480.059834,4789.659762,0.191306,0.079816
593,2026-08-17,60215,12028.060677,5226.072753,0.199752,0.086790
594,2026-08-18,60349,13393.997294,5547.357323,0.221942,0.091921
595,2026-08-19,60406,17314.530662,5745.743993,0.286636,0.095119


In [172]:
# Chart — blended (all segments) vs core (excl. new installs) ARPDAU IAP
compare_df_iap = arpdau_agg[['dt', 'arpdau_iap']].copy()
compare_df_iap['dt'] = pd.to_datetime(compare_df_iap['dt'])
compare_df_iap = compare_df_iap.merge(core_arpdau_agg[['dt', 'arpdau_iap']], on='dt', suffixes=('_blended', '_core'))

fig = px.line(
    compare_df_iap, x='dt', y=['arpdau_iap_blended', 'arpdau_iap_core'],
    title='ARPDAU IAP: blended (all segments) vs core (excl. new installs)',
    labels={'dt': 'Date', 'value': 'ARPDAU (IAP)', 'variable': ''},
    width=1400, height=500,
)
fig.show()

# Chart — blended (all segments) vs core (excl. new installs) ARPDAU Ad
compare_df_ad = arpdau_agg[['dt', 'arpdau_ad']].copy()
compare_df_ad['dt'] = pd.to_datetime(compare_df_ad['dt'])
compare_df_ad = compare_df_ad.merge(core_arpdau_agg[['dt', 'arpdau_ad']], on='dt', suffixes=('_blended', '_core'))

fig = px.line(
    compare_df_ad, x='dt', y=['arpdau_ad_blended', 'arpdau_ad_core'],
    title='ARPDAU Ad: blended (all segments) vs core (excl. new installs)',
    labels={'dt': 'Date', 'value': 'ARPDAU (Ad)', 'variable': ''},
    width=1400, height=500,
)
fig.show()

## ARPDAU IAP (Core) Forecast & Scenario Analysis

Same 7-step pipeline as the RRW/Loyalty sections above, applied to core ARPDAU (IAP revenue per DAU, excluding new installs). Two differences from those sections: **Section 4 uses a log-transform** (`fit_value_forecast`) instead of the logit-transform used for rate metrics, since ARPDAU is an unbounded positive value, not a [0,1] proportion — and **Section 6's uplift is unclipped** for the same reason (revenue has no natural 100% ceiling).

### 1. Data prep + trend/seasonality decomposition

In [173]:
# --- All tunable parameters for this section, in one place ---

metric_col = 'arpdau_iap'            # which core ARPDAU series to analyze

outlier_threshold = 7                # Sec 2: modified z-score threshold for flagging outliers (Iglewicz & Hoaglin standard)
min_segment_days = 14                  # Sec 3: minimum days between two detected change-points (regimes)
min_training_days = 90                 # Sec 4: forecast training window — walk back through regimes until >= this many days
sarimax_order = (1, 1, 1)              # Sec 4: SARIMAX (p, d, q) non-seasonal order
sarimax_seasonal_order = (1, 1, 1, 7)  # Sec 4: SARIMAX seasonal order, period=7 (weekly)
forecast_horizon_days = 90             # Sec 4: how many days ahead to forecast
backtest_days = 28                     # Sec 4: holdout days for the pre-forecast backtest
ci_alpha = 0.20                        # Sec 4: forecast interval width (0.20 -> 80% interval, ~P10/P90)
uplift_pct_arpdau_iap = 0.02    # Sec 6: relative uplift applied to the forecast curve (e.g. 0.02 = +2%)

In [174]:
# hide-output
arpdau_iap_series = core_arpdau_agg.set_index('dt')[metric_col].asfreq('D')
n_gaps = arpdau_iap_series.isna().sum()
arpdau_iap_series = arpdau_iap_series.interpolate()  # fill any single-day calendar gaps
arpdau_iap_series.name = metric_col

print(f"{metric_col}: {len(arpdau_iap_series)} days, {arpdau_iap_series.index.min().date()} -> {arpdau_iap_series.index.max().date()} ({n_gaps} interpolated gap-days)")

arpdau_iap: 597 days, 2025-01-01 -> 2026-08-20 (0 interpolated gap-days)


In [175]:
# hide-output
arpdau_iap_decomp, arpdau_iap_adf_stat, arpdau_iap_adf_p = decompose_series(arpdau_iap_series, period=7)
print(f"ADF test on STL residual: stat={arpdau_iap_adf_stat:.3f}, p={arpdau_iap_adf_p:.4f} -> {'stationary' if arpdau_iap_adf_p < 0.05 else 'NOT stationary'}")

ADF test on STL residual: stat=-10.293, p=0.0000 -> stationary


In [176]:
# Chart — STL decomposition: observed+trend / weekly seasonal / residual
plot_stl_decomposition(arpdau_iap_decomp, title=f'{metric_col} (core) — STL decomposition (weekly seasonality)').show()

### 2. Variance & outliers

Same MAD-based robust z-score as the RRW section (see the plain-language explanation there for how it works).

In [177]:
# hide-output
arpdau_iap_decomp['modified_z'], arpdau_iap_decomp['is_outlier'] = flag_robust_outliers(arpdau_iap_decomp['resid'], threshold=outlier_threshold)

cov = arpdau_iap_decomp['resid'].std() / arpdau_iap_decomp['value'].mean()
print(f"Residual std: {arpdau_iap_decomp['resid'].std():.5f}  |  MAD: {(arpdau_iap_decomp['resid'] - arpdau_iap_decomp['resid'].median()).abs().median():.5f}  |  series mean: {arpdau_iap_decomp['value'].mean():.4f}")
print(f"Coefficient of variation (resid std / series mean): {cov:.3%}")
print(f"Outliers flagged (|modified z| > {outlier_threshold}): {arpdau_iap_decomp['is_outlier'].sum()} / {len(arpdau_iap_decomp)} days")

arpdau_iap_decomp[arpdau_iap_decomp['is_outlier']][['value', 'resid', 'modified_z']]

Residual std: 0.06422  |  MAD: 0.00585  |  series mean: 0.2415
Coefficient of variation (resid std / series mean): 26.589%
Outliers flagged (|modified z| > 7): 84 / 597 days


,value,resid,modified_z
dt,,,
2025-01-17,0.341589,0.150099,17.282751
2025-01-24,0.498025,0.302464,34.860447
2025-01-25,0.296295,0.093050,10.701355
2025-01-31,0.187820,-0.159935,-18.484304
2025-02-07,0.180055,-0.193811,-22.392409
...,...,...,...
2026-07-25,0.365692,0.086042,9.892902
2026-08-04,0.314368,0.083725,9.625605
2026-08-09,0.330863,0.096439,11.092288


In [178]:
# Chart — observed series with flagged outlier days marked
plot_outliers(arpdau_iap_decomp, arpdau_iap_decomp['is_outlier'], title=f'{metric_col} (core) with robust-outlier days flagged').show()

### 3. Change-point detection (regime shifts)

Same PELT-based unsupervised detection as the RRW section above, on this metric's deseasonalized series.

In [179]:
# hide-output
arpdau_iap_changepoint_dates, arpdau_iap_changepoint_penalty = detect_changepoints(arpdau_iap_decomp['deseasonalized'], min_segment_days=min_segment_days)

print(f"Penalty: {arpdau_iap_changepoint_penalty:.2f}  |  Change-points detected: {len(arpdau_iap_changepoint_dates)}")
for d in arpdau_iap_changepoint_dates:
    print(' ', d.date())

Penalty: 6.39  |  Change-points detected: 2
  2025-05-15
  2026-01-30


In [180]:
# Chart — observed series with detected change-points marked
plot_changepoints(arpdau_iap_decomp, arpdau_iap_changepoint_dates, title=f'{metric_col} (core) with detected regime changes (dashed lines)').show()

### 4. Forecast (3 months ahead)

Same training-window walk-back as the RRW section, but the SARIMAX fit uses a **log-transform** (`fit_value_forecast`) instead of logit, since ARPDAU is an unbounded positive value, not a [0,1] rate. Revenue metrics are inherently noisier day-to-day than rate metrics (lumpier, driven by a smaller number of paying/high-value users), so expect a wider backtest error than the RRW/Loyalty sections.

In [181]:
# hide-output
# Walk back through detected regimes until we have >= min_training_days (set in Parameters above)
arpdau_iap_training_start = select_training_start(arpdau_iap_decomp.index, arpdau_iap_changepoint_dates, min_training_days=min_training_days)
arpdau_iap_train = arpdau_iap_series.loc[arpdau_iap_training_start:]
print(f"Training window: {arpdau_iap_training_start.date()} -> {arpdau_iap_decomp.index.max().date()} ({len(arpdau_iap_train)} days)")

Training window: 2026-01-30 -> 2026-08-20 (203 days)


In [182]:
# hide-output
arpdau_iap_backtest_mae, arpdau_iap_backtest_mape, arpdau_iap_forecast_df = fit_value_forecast(
    arpdau_iap_train, order=sarimax_order, seasonal_order=sarimax_seasonal_order,
    horizon=forecast_horizon_days, backtest_days=backtest_days, ci_alpha=ci_alpha,
)
print(f"Backtest (last {backtest_days} days of training window): MAE={arpdau_iap_backtest_mae:.5f}  MAPE={arpdau_iap_backtest_mape:.3%}")

Backtest (last 28 days of training window): MAE=0.05114  MAPE=17.837%


In [183]:
# hide-output
print(f"Forecast horizon: {arpdau_iap_forecast_df.index.min().date()} -> {arpdau_iap_forecast_df.index.max().date()}")
for h in [30, 60, 90]:
    row = arpdau_iap_forecast_df.iloc[h - 1]
    print(f"  +{h}d ({arpdau_iap_forecast_df.index[h-1].date()}): {row['forecast']:.4f}  [{row['low']:.4f}, {row['high']:.4f}]")

Forecast horizon: 2026-08-21 -> 2026-11-18
  +30d (2026-09-19): 0.3054  [0.2286, 0.4080]
  +60d (2026-10-19): 0.2449  [0.1826, 0.3285]
  +90d (2026-11-18): 0.2572  [0.1910, 0.3465]


### 5. Summary: history + forecast

In [184]:
# Chart — training window history + 90-day forecast with 80% band
plot_forecast_summary(
    arpdau_iap_series, arpdau_iap_train, arpdau_iap_forecast_df, arpdau_iap_training_start,
    title=f'{metric_col} (core) — history + {forecast_horizon_days}-day forecast', ci_alpha=ci_alpha,
).show()

In [185]:
# Summary table — forecast at key checkpoints
arpdau_iap_summary_table = arpdau_iap_forecast_df.iloc[[29, 59, 89]].reset_index().rename(columns={
    'dt': 'Date', 'forecast': 'P50 (forecast)', 'low': 'P10', 'high': 'P90',
})
arpdau_iap_summary_table.insert(0, 'Days out', [30, 60, 90])
arpdau_iap_summary_table[['P10', 'P50 (forecast)', 'P90']] = arpdau_iap_summary_table[['P10', 'P50 (forecast)', 'P90']].round(4)
arpdau_iap_summary_table

,Days out,Date,P50 (forecast),P10,P90
0,30,2026-09-19,0.3054,0.2286,0.4080
1,60,2026-10-19,0.2449,0.1826,0.3285
2,90,2026-11-18,0.2572,0.1910,0.3465


### 6. Scenario: what if core ARPDAU IAP gets a % uplift

Same standalone overlay approach as the RRW section — no upper clip here, since revenue has no natural 100% ceiling. Adjust `uplift_pct_arpdau_iap` to try different scenarios.

In [186]:
# hide-output
arpdau_iap_scenario_df = apply_uplift_scenario(arpdau_iap_forecast_df, uplift_pct_arpdau_iap, clip_upper=None)

print(f"Uplift scenario: {uplift_pct_arpdau_iap:+.1%} relative")
for h in [30, 60, 90]:
    base = arpdau_iap_scenario_df['forecast'].iloc[h - 1]
    up = arpdau_iap_scenario_df['forecast_uplift'].iloc[h - 1]
    print(f"  +{h}d: baseline={base:.4f}  uplifted={up:.4f}  (delta={up - base:+.4f})")

Uplift scenario: +2.0% relative
  +30d: baseline=0.3054  uplifted=0.3115  (delta=+0.0061)
  +60d: baseline=0.2449  uplifted=0.2498  (delta=+0.0049)
  +90d: baseline=0.2572  uplifted=0.2624  (delta=+0.0051)


In [187]:
# Chart — baseline forecast vs uplift scenario
plot_uplift_scenario(
    arpdau_iap_series, arpdau_iap_scenario_df, uplift_pct_arpdau_iap,
    title=f'{metric_col} (core) — baseline vs {uplift_pct_arpdau_iap:+.1%} uplift scenario', lookback_days=60,
).show()

### 7. Period comparison: Period 1 vs Period 2

Average core `metric_col` within each of the two shared periods defined at the top of the notebook, and the relative uplift between them.

In [188]:
# hide-output
arpdau_iap_period_comparison = compare_periods(arpdau_iap_series, period1_start, period1_end, period2_start, period2_end)

print(f"Period 1 ({period1_start} -> {period1_end}, n={arpdau_iap_period_comparison['period1_n']} days): avg {metric_col} = {arpdau_iap_period_comparison['period1_mean']:.4f}")
print(f"Period 2 ({period2_start} -> {period2_end}, n={arpdau_iap_period_comparison['period2_n']} days): avg {metric_col} = {arpdau_iap_period_comparison['period2_mean']:.4f}")
print(f"Uplift: {arpdau_iap_period_comparison['abs_diff']:+.4f}  ({arpdau_iap_period_comparison['pct_uplift']:+.1%} relative)")

Period 1 (2025-07-01 -> 2025-09-30, n=92 days): avg arpdau_iap = 0.2327
Period 2 (2026-01-01 -> 2026-08-31, n=232 days): avg arpdau_iap = 0.2725
Uplift: +0.0398  (+17.1% relative)


In [189]:
# Chart — Period 1 vs Period 2 average
plot_period_comparison(
    arpdau_iap_period_comparison, title=f'{metric_col} (core): Period 1 vs Period 2 average', y_label=metric_col,
).show()

## ARPDAU Ad (Core) Forecast & Scenario Analysis

Same 7-step pipeline as the RRW/Loyalty sections above, applied to core ARPDAU (Ad revenue per DAU, excluding new installs). Two differences from those sections: **Section 4 uses a log-transform** (`fit_value_forecast`) instead of the logit-transform used for rate metrics, since ARPDAU is an unbounded positive value, not a [0,1] proportion — and **Section 6's uplift is unclipped** for the same reason (revenue has no natural 100% ceiling).

### 1. Data prep + trend/seasonality decomposition

In [190]:
# --- All tunable parameters for this section, in one place ---

metric_col = 'arpdau_ad'            # which core ARPDAU series to analyze

outlier_threshold = 7                # Sec 2: modified z-score threshold for flagging outliers (Iglewicz & Hoaglin standard)
min_segment_days = 14                  # Sec 3: minimum days between two detected change-points (regimes)
min_training_days = 90                 # Sec 4: forecast training window — walk back through regimes until >= this many days
sarimax_order = (1, 1, 1)              # Sec 4: SARIMAX (p, d, q) non-seasonal order
sarimax_seasonal_order = (1, 1, 1, 7)  # Sec 4: SARIMAX seasonal order, period=7 (weekly)
forecast_horizon_days = 90             # Sec 4: how many days ahead to forecast
backtest_days = 28                     # Sec 4: holdout days for the pre-forecast backtest
ci_alpha = 0.20                        # Sec 4: forecast interval width (0.20 -> 80% interval, ~P10/P90)
uplift_pct_arpdau_ad = 0.02    # Sec 6: relative uplift applied to the forecast curve (e.g. 0.02 = +2%)

In [191]:
# hide-output
arpdau_ad_series = core_arpdau_agg.set_index('dt')[metric_col].asfreq('D')
n_gaps = arpdau_ad_series.isna().sum()
arpdau_ad_series = arpdau_ad_series.interpolate()  # fill any single-day calendar gaps
arpdau_ad_series.name = metric_col

print(f"{metric_col}: {len(arpdau_ad_series)} days, {arpdau_ad_series.index.min().date()} -> {arpdau_ad_series.index.max().date()} ({n_gaps} interpolated gap-days)")

arpdau_ad: 597 days, 2025-01-01 -> 2026-08-20 (0 interpolated gap-days)


In [192]:
# hide-output
arpdau_ad_decomp, arpdau_ad_adf_stat, arpdau_ad_adf_p = decompose_series(arpdau_ad_series, period=7)
print(f"ADF test on STL residual: stat={arpdau_ad_adf_stat:.3f}, p={arpdau_ad_adf_p:.4f} -> {'stationary' if arpdau_ad_adf_p < 0.05 else 'NOT stationary'}")

ADF test on STL residual: stat=-19.469, p=0.0000 -> stationary


In [193]:
# Chart — STL decomposition: observed+trend / weekly seasonal / residual
plot_stl_decomposition(arpdau_ad_decomp, title=f'{metric_col} (core) — STL decomposition (weekly seasonality)').show()

### 2. Variance & outliers

Same MAD-based robust z-score as the RRW section (see the plain-language explanation there for how it works).

In [194]:
# hide-output
arpdau_ad_decomp['modified_z'], arpdau_ad_decomp['is_outlier'] = flag_robust_outliers(arpdau_ad_decomp['resid'], threshold=outlier_threshold)

cov = arpdau_ad_decomp['resid'].std() / arpdau_ad_decomp['value'].mean()
print(f"Residual std: {arpdau_ad_decomp['resid'].std():.5f}  |  MAD: {(arpdau_ad_decomp['resid'] - arpdau_ad_decomp['resid'].median()).abs().median():.5f}  |  series mean: {arpdau_ad_decomp['value'].mean():.4f}")
print(f"Coefficient of variation (resid std / series mean): {cov:.3%}")
print(f"Outliers flagged (|modified z| > {outlier_threshold}): {arpdau_ad_decomp['is_outlier'].sum()} / {len(arpdau_ad_decomp)} days")

arpdau_ad_decomp[arpdau_ad_decomp['is_outlier']][['value', 'resid', 'modified_z']]

Residual std: 0.00370  |  MAD: 0.00068  |  series mean: 0.0847
Coefficient of variation (resid std / series mean): 4.371%
Outliers flagged (|modified z| > 7): 21 / 597 days


,value,resid,modified_z
dt,,,
2025-02-05,0.082266,0.007272,7.249806
2025-02-07,0.078221,0.008363,8.336930
2025-03-10,0.090360,0.009077,9.047357
2025-03-11,0.091216,0.008716,8.688127
2025-03-12,0.089321,0.008332,8.305677
2025-04-07,0.088974,0.007405,7.382453
2025-04-08,0.089548,0.009035,9.005502
2025-05-07,0.095494,0.010966,10.928888
2025-05-08,0.091475,0.010960,10.922927


In [195]:
# Chart — observed series with flagged outlier days marked
plot_outliers(arpdau_ad_decomp, arpdau_ad_decomp['is_outlier'], title=f'{metric_col} (core) with robust-outlier days flagged').show()

### 3. Change-point detection (regime shifts)

Same PELT-based unsupervised detection as the RRW section above, on this metric's deseasonalized series.

In [196]:
# hide-output
arpdau_ad_changepoint_dates, arpdau_ad_changepoint_penalty = detect_changepoints(arpdau_ad_decomp['deseasonalized'], min_segment_days=min_segment_days)

print(f"Penalty: {arpdau_ad_changepoint_penalty:.2f}  |  Change-points detected: {len(arpdau_ad_changepoint_dates)}")
for d in arpdau_ad_changepoint_dates:
    print(' ', d.date())

Penalty: 6.39  |  Change-points detected: 7
  2025-01-15
  2025-03-06
  2025-06-19
  2025-11-21
  2026-02-04
  2026-04-30
  2026-07-09


In [197]:
# Chart — observed series with detected change-points marked
plot_changepoints(arpdau_ad_decomp, arpdau_ad_changepoint_dates, title=f'{metric_col} (core) with detected regime changes (dashed lines)').show()

### 4. Forecast (3 months ahead)

Same training-window walk-back as the RRW section, but the SARIMAX fit uses a **log-transform** (`fit_value_forecast`) instead of logit, since ARPDAU is an unbounded positive value, not a [0,1] rate. Revenue metrics are inherently noisier day-to-day than rate metrics (lumpier, driven by a smaller number of paying/high-value users), so expect a wider backtest error than the RRW/Loyalty sections.

In [198]:
# hide-output
# Walk back through detected regimes until we have >= min_training_days (set in Parameters above)
arpdau_ad_training_start = select_training_start(arpdau_ad_decomp.index, arpdau_ad_changepoint_dates, min_training_days=min_training_days)
arpdau_ad_train = arpdau_ad_series.loc[arpdau_ad_training_start:]
print(f"Training window: {arpdau_ad_training_start.date()} -> {arpdau_ad_decomp.index.max().date()} ({len(arpdau_ad_train)} days)")

Training window: 2026-04-30 -> 2026-08-20 (113 days)


In [199]:
# hide-output
arpdau_ad_backtest_mae, arpdau_ad_backtest_mape, arpdau_ad_forecast_df = fit_value_forecast(
    arpdau_ad_train, order=sarimax_order, seasonal_order=sarimax_seasonal_order,
    horizon=forecast_horizon_days, backtest_days=backtest_days, ci_alpha=ci_alpha,
)
print(f"Backtest (last {backtest_days} days of training window): MAE={arpdau_ad_backtest_mae:.5f}  MAPE={arpdau_ad_backtest_mape:.3%}")

Backtest (last 28 days of training window): MAE=0.00322  MAPE=3.788%


In [200]:
# hide-output
print(f"Forecast horizon: {arpdau_ad_forecast_df.index.min().date()} -> {arpdau_ad_forecast_df.index.max().date()}")
for h in [30, 60, 90]:
    row = arpdau_ad_forecast_df.iloc[h - 1]
    print(f"  +{h}d ({arpdau_ad_forecast_df.index[h-1].date()}): {row['forecast']:.4f}  [{row['low']:.4f}, {row['high']:.4f}]")

Forecast horizon: 2026-08-21 -> 2026-11-18
  +30d (2026-09-19): 0.1000  [0.0660, 0.1516]
  +60d (2026-10-19): 0.1159  [0.0457, 0.2935]
  +90d (2026-11-18): 0.1361  [0.0286, 0.6482]


### 5. Summary: history + forecast

In [201]:
# Chart — training window history + 90-day forecast with 80% band
plot_forecast_summary(
    arpdau_ad_series, arpdau_ad_train, arpdau_ad_forecast_df, arpdau_ad_training_start,
    title=f'{metric_col} (core) — history + {forecast_horizon_days}-day forecast', ci_alpha=ci_alpha,
).show()

In [202]:
# Summary table — forecast at key checkpoints
arpdau_ad_summary_table = arpdau_ad_forecast_df.iloc[[29, 59, 89]].reset_index().rename(columns={
    'dt': 'Date', 'forecast': 'P50 (forecast)', 'low': 'P10', 'high': 'P90',
})
arpdau_ad_summary_table.insert(0, 'Days out', [30, 60, 90])
arpdau_ad_summary_table[['P10', 'P50 (forecast)', 'P90']] = arpdau_ad_summary_table[['P10', 'P50 (forecast)', 'P90']].round(4)
arpdau_ad_summary_table

,Days out,Date,P50 (forecast),P10,P90
0,30,2026-09-19,0.1000,0.0660,0.1516
1,60,2026-10-19,0.1159,0.0457,0.2935
2,90,2026-11-18,0.1361,0.0286,0.6482


### 6. Scenario: what if core ARPDAU Ad gets a % uplift

Same standalone overlay approach as the RRW section — no upper clip here, since revenue has no natural 100% ceiling. Adjust `uplift_pct_arpdau_ad` to try different scenarios.

In [203]:
# hide-output
arpdau_ad_scenario_df = apply_uplift_scenario(arpdau_ad_forecast_df, uplift_pct_arpdau_ad, clip_upper=None)

print(f"Uplift scenario: {uplift_pct_arpdau_ad:+.1%} relative")
for h in [30, 60, 90]:
    base = arpdau_ad_scenario_df['forecast'].iloc[h - 1]
    up = arpdau_ad_scenario_df['forecast_uplift'].iloc[h - 1]
    print(f"  +{h}d: baseline={base:.4f}  uplifted={up:.4f}  (delta={up - base:+.4f})")

Uplift scenario: +2.0% relative
  +30d: baseline=0.1000  uplifted=0.1020  (delta=+0.0020)
  +60d: baseline=0.1159  uplifted=0.1182  (delta=+0.0023)
  +90d: baseline=0.1361  uplifted=0.1388  (delta=+0.0027)


In [204]:
# Chart — baseline forecast vs uplift scenario
plot_uplift_scenario(
    arpdau_ad_series, arpdau_ad_scenario_df, uplift_pct_arpdau_ad,
    title=f'{metric_col} (core) — baseline vs {uplift_pct_arpdau_ad:+.1%} uplift scenario', lookback_days=60,
).show()

### 7. Period comparison: Period 1 vs Period 2

Average core `metric_col` within each of the two shared periods defined at the top of the notebook, and the relative uplift between them.

In [205]:
# hide-output
arpdau_ad_period_comparison = compare_periods(arpdau_ad_series, period1_start, period1_end, period2_start, period2_end)

print(f"Period 1 ({period1_start} -> {period1_end}, n={arpdau_ad_period_comparison['period1_n']} days): avg {metric_col} = {arpdau_ad_period_comparison['period1_mean']:.4f}")
print(f"Period 2 ({period2_start} -> {period2_end}, n={arpdau_ad_period_comparison['period2_n']} days): avg {metric_col} = {arpdau_ad_period_comparison['period2_mean']:.4f}")
print(f"Uplift: {arpdau_ad_period_comparison['abs_diff']:+.4f}  ({arpdau_ad_period_comparison['pct_uplift']:+.1%} relative)")

Period 1 (2025-07-01 -> 2025-09-30, n=92 days): avg arpdau_ad = 0.0791
Period 2 (2026-01-01 -> 2026-08-31, n=232 days): avg arpdau_ad = 0.0903
Uplift: +0.0112  (+14.2% relative)


In [206]:
# Chart — Period 1 vs Period 2 average
plot_period_comparison(
    arpdau_ad_period_comparison, title=f'{metric_col} (core): Period 1 vs Period 2 average', y_label=metric_col,
).show()

# Dedicated Segment: Player Composition by DSI Segment

In [207]:
dsi_agg = loyalty_data[['dt', 'loyalty_segment', 'dsi_segment', 'user_id']][loyalty_data.loyalty_segment == '1. 26-28 (dedicated)'].copy()

dsi_agg = dsi_agg.groupby(['dt', 'dsi_segment']).agg(
    unique_users = ('user_id', 'nunique'),
).reset_index().sort_values(by=['dt', 'dsi_segment'])

# Add total dedicated-segment users per day
dsi_agg['total_users_per_day'] = dsi_agg.groupby('dt')['unique_users'].transform('sum')

# Add percentage column
dsi_agg['percentage'] = (dsi_agg['unique_users'] / dsi_agg['total_users_per_day'])

dsi_agg

,dt,dsi_segment,unique_users,total_users_per_day,percentage
0,2025-01-01,D007-D027,206,88523,0.002327
1,2025-01-01,D028-D090,16539,88523,0.186833
2,2025-01-01,D091-D181,14971,88523,0.169120
3,2025-01-01,D182-D363,15357,88523,0.173480
4,2025-01-01,D364+,41450,88523,0.468240
...,...,...,...,...,...
2980,2026-08-20,D007-D027,23,34801,0.000661
2981,2026-08-20,D028-D090,2626,34801,0.075458
2982,2026-08-20,D091-D181,1976,34801,0.056780
2983,2026-08-20,D182-D363,4299,34801,0.123531


In [208]:
# Create a line chart showing DSI segment composition over time, within the dedicated loyalty segment
fig = px.line(
    dsi_agg,
    x='dt',
    y='percentage',
    color='dsi_segment',
    title='Dedicated Segment — Player % by DSI Segment Over Time',
    labels={'dt': 'Date', 'percentage': '% of Dedicated Segment Players', 'dsi_segment': 'DSI Segment'},
    markers=False,
    height=600,
    width=1500
)

fig.show()